# Batch Normalization — Accelerating Deep Network Training by Reducing Internal Covariate Shift (2015)

_Papers · Foundations & Optimization_

**Normalize each layer's inputs over the mini-batch, then learn a scale and shift — so deep nets train far faster.**

---

This notebook is a practice scaffold for the **Batch Normalization — Accelerating Deep Network Training by Reducing Internal Covariate Shift (2015)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch, torch.nn as nn

torch.manual_seed(0)

class MyBatchNorm1d(nn.Module):
    """BatchNorm1d from scratch — Algorithm 1 of Ioffe & Szegedy (2015)."""
    def __init__(self, num_features, eps=1e-5, momentum=0.1):
        super().__init__()
        self.eps, self.momentum = eps, momentum
        self.gamma = nn.Parameter(torch.ones(num_features))   # learned scale
        self.beta  = nn.Parameter(torch.zeros(num_features))  # learned shift
        # running (population) stats — buffers, not trained
        self.register_buffer("running_mean", torch.zeros(num_features))
        self.register_buffer("running_var",  torch.ones(num_features))

    def forward(self, x):                      # x: (batch, features)
        if self.training:
            mu  = x.mean(dim=0)                                  # mini-batch mean  (Alg.1 line 1)
            var = x.var(dim=0, unbiased=False)                  # mini-batch var, biased /m (line 2)
            # update running stats with the UNBIASED variance (matches PyTorch)
            m = x.shape[0]
            with torch.no_grad():
                self.running_mean = (1-self.momentum)*self.running_mean + self.momentum*mu
                ubvar = x.var(dim=0, unbiased=True)             # /(m-1)
                self.running_var  = (1-self.momentum)*self.running_var  + self.momentum*ubvar
        else:
            mu, var = self.running_mean, self.running_var       # inference: fixed pop stats (Alg.2)
        xhat = (x - mu) / torch.sqrt(var + self.eps)            # normalize        (line 3)
        return self.gamma * xhat + self.beta                    # scale & shift    (line 4)

# ---- THE ORACLE: my BN must equal nn.BatchNorm1d, train AND eval ----
F = 4
x = torch.randn(8, F)
mine = MyBatchNorm1d(F)
ref  = nn.BatchNorm1d(F)          # same default eps=1e-5, momentum=0.1, gamma=1, beta=0

mine.train(); ref.train()
ok_train = torch.allclose(mine(x), ref(x), atol=1e-6)
# run a few train steps so running stats accumulate identically, then compare eval
for _ in range(5):
    xb = torch.randn(8, F); mine(xb); ref(xb)
mine.eval(); ref.eval()
xt = torch.randn(3, F)
ok_eval = torch.allclose(mine(xt), ref(xt), atol=1e-6)
print("allclose train:", ok_train)   # expect True
print("allclose eval :", ok_eval)    # expect True

# ---- recompute the worked example: batch [2,4,6], gamma=2, beta=1 ----
bn = MyBatchNorm1d(1); bn.train()
with torch.no_grad():
    bn.gamma.fill_(2.0); bn.beta.fill_(1.0)
xe = torch.tensor([[2.],[4.],[6.]])
print("worked example y:", bn(xe).squeeze().tolist())  # ~ [-1.449, 1.0, 3.449]

# ---- drop it into a 2-layer net ----
net = nn.Sequential(nn.Linear(F, 16), MyBatchNorm1d(16), nn.ReLU(), nn.Linear(16, 2))
print("net output shape:", net(torch.randn(5, F)).shape)  # torch.Size([5, 2])

## Visualize the data & results

_Train the same tiny 2-layer classifier on the same toy data at the same high learning rate, with vs without a BatchNorm layer — does BN converge faster and more stably?_

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

# toy 2-class data; scale features unevenly to create covariate-shift-like conditioning
N = 400
X0 = rng.normal([-2,-2], 1.0, (N//2, 2)); X1 = rng.normal([2,2], 1.0, (N//2, 2))
X = np.vstack([X0, X1]).astype(np.float64) * np.array([6.0, 0.3])
y = np.array([0]*(N//2) + [1]*(N//2))
p = rng.permutation(N); X, y = X[p], y[p]

def init(h=16):
    r = np.random.default_rng(1)
    return dict(W1=r.normal(0,0.6,(2,h)), b1=np.zeros(h), g=np.ones(h), be=np.zeros(h),
                W2=r.normal(0,0.6,(h,1)), b2=np.zeros(1))

def sig(z): return 1/(1+np.exp(-np.clip(z,-30,30)))

def train(use_bn, steps=40, lr=0.6, h=16, eps=1e-5):
    P = init(h); losses = []
    for t in range(steps):
        z1 = X @ P['W1'] + P['b1']
        if use_bn:                                  # BatchNorm on the hidden pre-activations
            mu = z1.mean(0); var = z1.var(0)
            zhat = (z1 - mu) / np.sqrt(var + eps)
            z1n = P['g'] * zhat + P['be']
        else:
            z1n = z1
        a1 = np.tanh(z1n); z2 = a1 @ P['W2'] + P['b2']
        out = np.clip(sig(z2).ravel(), 1e-7, 1-1e-7)
        loss = -np.mean(y*np.log(out) + (1-y)*np.log(1-out)); losses.append(round(float(loss),4))
        # backward
        dz2 = (out - y).reshape(-1,1)/N; dW2 = a1.T @ dz2; db2 = dz2.sum(0)
        da1 = dz2 @ P['W2'].T; dz1n = da1 * (1 - a1**2)
        if use_bn:
            dg = (dz1n*zhat).sum(0); dbe = dz1n.sum(0); dzhat = dz1n*P['g']; m = N
            dvar = (dzhat*(z1-mu)*-0.5*(var+eps)**-1.5).sum(0)
            dmu  = (dzhat*-1/np.sqrt(var+eps)).sum(0) + dvar*(-2*(z1-mu)).mean(0)
            dz1  = dzhat/np.sqrt(var+eps) + dvar*2*(z1-mu)/m + dmu/m
            P['g'] -= lr*dg; P['be'] -= lr*dbe
        else:
            dz1 = dz1n
        dW1 = X.T @ dz1; db1 = dz1.sum(0)
        P['W1'] -= lr*dW1; P['b1'] -= lr*db1; P['W2'] -= lr*dW2; P['b2'] -= lr*db2
    return losses

with_bn = train(True); no_bn = train(False)
print("with BN  final loss:", with_bn[-1])   # ~0.0338
print("no  BN   final loss:", no_bn[-1])      # ~0.0598

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Normalize the batch [10, 20, 30] for one feature, then apply $\gamma=3,\beta=5$. (Ignore $\epsilon$.)

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Mean: $\mu_B=(10+20+30)/3=20$. — _Center of the batch._
- Variance: $\sigma^2_B=((10-20)^2+0+(30-20)^2)/3=(100+0+100)/3=66.67$; std $\approx 8.165$. — _Spread; biased (divide by 3)._
- Normalize: $[(10-20)/8.165,\,0,\,(30-20)/8.165]=[-1.225,0,1.225]$. — _Mean 0, variance ~1._
- Scale/shift: $3\cdot[-1.225,0,1.225]+5=[1.325,5,8.675]$. — _Learned $\gamma,\beta$ pick the final spread and center._

**Answer:** $\hat{x}=[-1.225,0,1.225]$, output $=[1.325,5,8.675]$. Note the normalized values are identical to the [2,4,6] example &mdash; BN is invariant to the input's original scale and shift.

</details>

**Problem 2.** Ablation: in the CODE's 2-layer net, remove the BatchNorm layer and raise the learning rate. What do you expect to happen to training?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Delete the BN layer so the first linear layer's raw outputs feed straight into the activation. — _Removes the per-step re-centering/re-scaling._
- Train with the higher learning rate and watch the loss curve. — _BN's main benefit is tolerating larger steps._
- Compare to the BN run (the CODEVIZ chart). — _Same data, same rate &mdash; only BN differs._

**Answer:** Without BN the run is less stable early and converges slower to a higher loss; in our small run (CODEVIZ) BN reaches loss ~0.034 vs ~0.060 without, in the same 40 steps. With a high enough rate the no-BN net can diverge entirely. This reproduces the paper's qualitative claim that BN allows higher learning rates and faster convergence.

</details>

**Problem 3.** Why does calling $\gamma$ and $\beta$ "learnable" matter &mdash; couldn't we just always output mean 0, variance 1?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Imagine the next layer is a sigmoid, which is most useful around 0 but should sometimes operate elsewhere. — _Fixing variance 1 might be wrong for the task._
- Note that with $\gamma,\beta$ trainable, the network can recover any mean/spread, including the original inputs. — _BN adds no constraint it cannot undo._

**Answer:** Forcing mean 0 / variance 1 could throw away useful structure (e.g. push a sigmoid into its linear region). Letting $\gamma,\beta$ be learned means BN can normalize when that helps and undo it when it does not, so it only stabilizes &mdash; it never removes the layer's representational power.

</details>